In [0]:
%run ../config

In [0]:
from pyspark.sql import functions as F

# Ensure Silver Table Exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_catalog}.order_details_clean (
    order_id INT,
    order_detail_id INT,
    amount INT,
    unit_price DOUBLE,
    total_price DOUBLE,
    item_id INT,
    item_code INT
)      
""")


# Read Bronze table
df = spark.read.table(f"{bronze_catalog}.order_details")


# Silver Transformations
df = (
    df
    # Standardize Unit Price
    .withColumn(
        "unit_price",
        F.regexp_replace(F.col("unit_price"), ",", ".").cast("double")
    )
    .withColumn(
        "unit_price",
        F.round(F.col("unit_price"), 2)
    )

    # Standardize Total Price
    .withColumn(
        "total_price",
        F.regexp_replace(F.col("total_price"), ",", ".").cast("double")
    )
    .withColumn(
        "total_price",
        F.round(F.col("total_price"), 2)
    )

    # Cast Item Code to INT
    .withColumn(
        "item_code",
        F.col("item_code").cast("int")
    )
)


# Final Schema
silver_df = (
    df.select(
        "order_id",
        "order_detail_id",
        "amount",
        "unit_price",
        "total_price",
        "item_id",
        "item_code"
    )
)


# Merge into silver table
silver_df.createOrReplaceTempView("staging_order_details")

spark.sql(f"""
MERGE INTO {silver_catalog}.order_details_clean t
USING staging_order_details s 
ON t.order_id = s.order_id 
AND t.order_detail_id = s.order_detail_id

WHEN MATCHED AND (
    t.amount      <> s.amount OR
    t.unit_price  <> s.unit_price OR
    t.total_price <> s.total_price OR
    t.item_id     <> s.item_id OR
    t.item_code   <> s.item_code
)
THEN UPDATE SET
    t.amount      = s.amount,
    t.unit_price  = s.unit_price,
    t.total_price = s.total_price,
    t.item_id     = s.item_id,
    t.item_code   = s.item_code

WHEN NOT MATCHED THEN
INSERT (
   order_id,
   order_detail_id,
   amount,
   unit_price,
   total_price,
   item_id,
   item_code 
)
VALUES (
    s.order_id,
    s.order_detail_id,
    s.amount,
    s.unit_price,
    s.total_price,
    s.item_id,
    s.item_code
)
""")


# Validate that the Silver table was written successfully
count = spark.table(f"{silver_catalog}.order_details_clean").count()

# Ensure table is not empty
assert count > 0, "order_details_clean table is empty"

# Log result to job output
print(f"order_details_clean validation passed: {count:,} records")